#Pandas Data Reshaping

##What This Notebook Covers

####In this notebook, we will master data reshaping in Pandas.

##Topics Covered
###**1. Pivot Tables (pivot_table)**
####Transform data into summary tables using aggregation
### **2. Pivot vs Pivot Table**
####Understanding duplicates and choosing the right method
### **3 Melt (Wide → Long Format)**
####Converting data into a format suitable for analysis & ML
### **4. Stack & Unstack**
####Reshaping data using index-based transformations
###**5. MultiIndex Basics**
####Working with hierarchical indexing
### **6. Wide vs Long Data Formats**
####Understanding when and why to reshape data
###**7. GroupBy + Unstack**
####Alternative way to create pivot-like summaries
### **8. Handling Missing Data After Reshaping**
####Managing NaN values effectively

#**1. Pivot Table**
##1. What is a Pivot Table?
###A pivot table reshapes your data into a summary table
####Pivot tables in pandas allow summarizing large datasets by reshaping them into structured tables using aggregation functions.
####Unlike pivot(), pivot_table() can handle duplicate entries by applying aggregation, making it more suitable for real-world datasets.”
##2. Basic Syntax
`pd.pivot_table(df, values="column", index="row", columns="col", aggfunc="sum")`

##3. Create Dataset

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "City": ["Lahore","Lahore","Karachi","Karachi","Islamabad","Islamabad","Lahore","Karachi"],
    "Product": ["A","B","A","B","A","B","A","B"],
    "Sales": [100,200,150,300,200,250,120,180]
})

df

,City,Product,Sales
0,Lahore,A,100
1,Lahore,B,200
2,Karachi,A,150
3,Karachi,B,300
4,Islamabad,A,200
5,Islamabad,B,250
6,Lahore,A,120
7,Karachi,B,180


##4. Basic Pivot Table

In [ ]:
pivot = pd.pivot_table(df, values='Sales', index='City',columns='Product')
pivot

Product,A,B
City,,
Islamabad,200.0,250.0
Karachi,150.0,240.0
Lahore,110.0,200.0


###Why values are different?

####Because default aggregation is:

`aggfunc = "mean"`

##5. Aggregation Functions
###sum

In [ ]:
df

,City,Product,Sales
0,Lahore,A,100
1,Lahore,B,200
2,Karachi,A,150
3,Karachi,B,300
4,Islamabad,A,200
5,Islamabad,B,250
6,Lahore,A,120
7,Karachi,B,180


In [ ]:
pd.pivot_table(df, values='Sales', columns='Product', index='City',aggfunc="sum")

Product,A,B
City,,
Islamabad,200,250
Karachi,150,480
Lahore,220,200


###Count

In [ ]:
pd.pivot_table(df, values='Sales', columns='Product', index='City',aggfunc="count")

Product,A,B
City,,
Islamabad,1,1
Karachi,1,2
Lahore,2,1


###Mean

In [ ]:
pd.pivot_table(df, values='Sales', columns='Product', index='City',aggfunc="mean")

Product,A,B
City,,
Islamabad,200.0,250.0
Karachi,150.0,240.0
Lahore,110.0,200.0


###Pivot table ALWAYS aggregates data

##6. Multiple Aggregations

In [ ]:
pivot= pd.pivot_table(df, values='Sales', index='City', columns='Product', aggfunc=["sum", "mean"])
pivot

sum        mean       
Product      A    B      A      B
City                             
Islamabad  200  250  200.0  250.0
Karachi    150  480  150.0  240.0
Lahore     220  200  110.0  200.0

##7. fill_value (Handling NaN)

In [ ]:
pd.pivot_table(df, values='Sales', columns ='Product', index= "City", aggfunc=['sum','mean'], fill_value=0)

sum        mean       
Product      A    B      A      B
City                             
Islamabad  200  250  200.0  250.0
Karachi    150  480  150.0  240.0
Lahore     220  200  110.0  200.0

NaN = confusing

0 = meaningful

##8. margins=True
###Margins means Totals

In [ ]:
pd.pivot_table(df, values="Sales", index="City", columns="Product", aggfunc= 'sum', margins=True)


Product,A,B,All
City,,,
Islamabad,200,250,450
Karachi,150,480,630
Lahore,220,200,420
All,570,930,1500


###Output includes:
####All (row + column totals)

##9.Handling Duplicates
###Problem with Pivot

In [ ]:
#df.pivot(index="City", columns="Product", values="Sales")

####This will FAIL if duplicates exist
###Because:

####Lahore + A appears multiple times
###Why pivot_table works?

###Because it:

####aggregates duplicates automatically

#**2. Pivot vs Pivot Table**
####Use pivot_table() in real-world
####Use pivot() only when data is perfectly clean

##1. Create Dataset

In [ ]:

df = pd.DataFrame({
    "City": ["Lahore","Lahore","Karachi","Karachi","Lahore"],
    "Product": ["A","A","A","B","B"],
    "Sales": [100,150,200,300,250]
})

df

,City,Product,Sales
0,Lahore,A,100
1,Lahore,A,150
2,Karachi,A,200
3,Karachi,B,300
4,Lahore,B,250


##2. Using Pivot() (This will Fail)

In [ ]:
#df.pivot(index= 'City', columns='Products', values='Sales')

###Raise Error:
####ValueError: Index contains duplicate entries, cannot reshape

###Why did it fail?

###Because:

####City	Product	Count
####Lahore	A	2 times

####pivot() expects:
####ONE unique value per combination

##3. Using Pivot_table()

In [ ]:
pd.pivot_table(df, columns = 'Product', index= 'City', values='Sales', aggfunc=['sum','mean'], fill_value=0, margins=True)

sum              mean                   
Product    A    B   All      A      B         All
City                                             
Karachi  200  300   500  200.0  300.0  250.000000
Lahore   250  250   500  125.0  250.0  166.666667
All      450  550  1000  150.0  275.0  200.000000

###What happened?

###pivot_table():

####Detected duplicates
####Applied aggregation (sum)
####Returned correct result

##4. When to Use pivot()

###Use ONLY when:

####No duplicates exist
####Data is already clean

###Here Data is clean and perfect for Pivot

In [ ]:
df_unique = pd.DataFrame({
    "City": ["Lahore","Karachi"],
    "Product": ["A","B"],
    "Sales": [100,200]
})

df_unique.pivot(index="City", columns="Product", values="Sales")

Product,A,B
City,,
Karachi,NaN,200.0
Lahore,100.0,NaN


##5. When to Use pivot_table() (REAL WORLD)

###Use when:

####Data has duplicates
####You need aggregation
####Working with real datasets

####*pivot() requires unique index-column combinations, while pivot_table() can handle duplicates by applying aggregation functions like sum or mean.*





#**3. Melt (Wide → Long Transformation)**
####*pd.melt() is used to transform wide-format data into long format, making it suitable for analysis, visualization, and machine learning. It allows flexible restructuring by defining identifier variables (id_vars) and value variables (value_vars).*
###1. What is Melt?

####Melt converts data from:

###Wide Format → Long Format
####Example
####Wide Format (Human-friendly)
|Name|	Jan|	Feb|
|----|-----|-----|
|Ali |	100|	200|
|Sara|	150|	250|
####Long Format (ML-friendly)
|Name|	Month|	Sales|
|----|-------|-------|
|Ali|	Jan|	100|
|Ali|	Feb|	200|
|Sara|	Jan|	150|
|Sara|	Feb|	250|

####Melt “unpivots” columns into rows

##2. Basic Syntax
`pd.melt(df, id_vars=..., value_vars=...)`

##3. Create Dataset

In [ ]:

df = pd.DataFrame({
    "Name": ["Ali", "Sara"],
    "Jan": [100, 150],
    "Feb": [200, 250]
})

df

,Name,Jan,Feb
0,Ali,100,200
1,Sara,150,250


##4. Basic Melt

In [ ]:
melted = pd.melt(df, id_vars= 'Name')
melted

,Name,variable,value
0,Ali,Jan,100
1,Sara,Jan,150
2,Ali,Feb,200
3,Sara,Feb,250


##5. id_vars

###Columns to KEEP fixed

`id_vars="Name"`

####These do NOT change
####They identify the data

##6. value_vars

###Columns to UNPIVOT
`value_vars=["Jan", "Feb"]`
####If not given → pandas uses all remaining columns


##7. Rename Columns

In [ ]:
pd.melt(df, id_vars="Name", var_name='Month', value_name='Sales')

,Name,Month,Sales
0,Ali,Jan,100
1,Sara,Jan,150
2,Ali,Feb,200
3,Sara,Feb,250


In [ ]:
pd.melt(df, id_vars="Name", value_vars=['Jan',"Feb"], var_name='Month',value_name='Sales' )

,Name,Month,Sales
0,Ali,Jan,100
1,Sara,Jan,150
2,Ali,Feb,200
3,Sara,Feb,250


###Always rename:

####variable → meaningful name
####value → meaningful name

##8. Why Long Format is Needed
###Machine Learning Needs This

###Libraries like:

####scikit-learn
####seaborn

Prefer LONG format

##Wide Format Problem
###Jan, Feb → separate columns

####Hard to analyze trends

##Long Format Advantage
###Month → single column

###Easy to:

####1. groupby
####2. visualize
####3. model

###Melt increases row
### Reduces columns

#**4. Stack & Unstack (Index Reshaping)**

##1. What is Stack & Unstack?



####* Stack → “Compress columns into rows” (Wide → Long)
####* Unstack → “Spread rows into columns” (Long → Wide)

####Think of them as pivoting vertically or horizontally using the index structure.

####* .stack() → moves column labels into row index
####* .unstack() → moves row index into column labels

##2. Basic Example

In [ ]:
df = pd.DataFrame({
    "City": ["Lahore", "Lahore", "Karachi", "Karachi"],
    "Product": ["A", "B", "A", "B"],
    "Sales": [100, 200, 150, 250],
    "Profit": [30, 50, 20, 70]
})
df


,City,Product,Sales,Profit
0,Lahore,A,100,30
1,Lahore,B,200,50
2,Karachi,A,150,20
3,Karachi,B,250,70


In [ ]:
pivot = pd.pivot_table(df, index="City", columns="Product", values=["Sales","Profit"])
pivot

Profit        Sales       
Product      A     B      A      B
City                              
Karachi   20.0  70.0  150.0  250.0
Lahore    30.0  50.0  100.0  200.0

##3. Stacking

In [ ]:
stacked = pivot.stack()
stacked

/tmp/ipykernel_706/235426745.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked = pivot.stack()


Profit  Sales
City    Product               
Karachi A          20.0  150.0
        B          70.0  250.0
Lahore  A          30.0  100.0
        B          50.0  200.0

####MultiIndex now has City & Product
####Columns (Sales, Profit) are now values

4. Unstacking

In [ ]:
unstacked = stacked.unstack(level=-1)
unstacked

Profit        Sales       
Product      A     B      A      B
City                              
Karachi   20.0  70.0  150.0  250.0
Lahore    30.0  50.0  100.0  200.0

####Brings it back to wide format (columns are Product again):
####.unstack() is basically the inverse of .stack()
####You can choose the level of the MultiIndex to unstack

##5. Stack & Unstack with Single-Level Index

In [ ]:
df_simple = pd.DataFrame({
    "A": [1, 2],
    "B": [3, 4]
}, index=["X", "Y"])
df_simple

,A,B
X,1,3
Y,2,4


In [ ]:
stacking= df_simple.stack()
stacking

X  A    1
   B    3
Y  A    2
   B    4
dtype: int64

####Stack → Series with MultiIndex (X,A, X,B, etc.)

In [ ]:
unstacking = stacking.unstack()
unstacking

,A,B
X,1,3
Y,2,4


####Unstack → back to original DataFrame

###Works with any index, but more useful with MultiIndex

##6. Handling Missing Values

####When you stack or unstack, missing combinations appear as NaN:

In [ ]:
df2 = pd.DataFrame({
    "City": ["Lahore", "Karachi"],
    "Product": ["A", "B"],
    "Sales": [100, 200]
})
df2

,City,Product,Sales
0,Lahore,A,100
1,Karachi,B,200


In [ ]:
pivot2 = pd.pivot_table(df2, index='City', columns= 'Product', values='Sales')
pivot2

Product,A,B
City,,
Karachi,NaN,200.0
Lahore,100.0,NaN


###Dropna

In [ ]:
stacked2 = pivot2.stack(dropna=False)
stacked2

/tmp/ipykernel_706/1692122053.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  stacked2 = pivot2.stack(dropna=False)


City     Product
Karachi  A            NaN
         B          200.0
Lahore   A          100.0
         B            NaN
dtype: float64

###Fillna

In [ ]:
stacked2_filled = stacked2.fillna(0)
stacked2_filled

City     Product
Karachi  A            0.0
         B          200.0
Lahore   A          100.0
         B            0.0
dtype: float64

##7. Level Parameter
####* .stack(level=1) → stack only level 1 of columns
####* .unstack(level=0) → unstack top-level index
####Essential when working with multi-level pivot tables

In [ ]:
unstacking1 = stacked2.unstack(level=0)
unstacking1

City,Karachi,Lahore
Product,,
A,NaN,100.0
B,200.0,NaN


####Here level = -1 unstacked top level index that is city

In [ ]:
unstacking2 = stacked2.unstack(level=-1)
unstacking2

Product,A,B
City,,
Karachi,NaN,200.0
Lahore,100.0,NaN


####Here level = 0 unstacked second level index that is product

####Visualization → Libraries like Seaborn prefer long format

##**5. MultiIndex**
##1. What is MultiIndex?

####A MultiIndex (or hierarchical index) is an index with multiple levels.

####1. Useful for high-dimensional data in 2D DataFrame.
####2. Each row is identified by more than one label.
####3. Commonly created by pivot_table(), groupby(), or concat(keys=...).

In [ ]:
data = {
    "City": ["Lahore", "Lahore", "Karachi", "Karachi"],
    "Month": ["Jan", "Feb", "Jan", "Feb"],
    "Sales": [100, 150, 200, 250]
}

df = pd.DataFrame(data)
pivot_df = df.pivot_table(index=["City", "Month"], values="Sales")
pivot_df

Sales
City    Month       
Karachi Feb    250.0
        Jan    200.0
Lahore  Feb    150.0
        Jan    100.0

##2. Creating MultiIndex
####a) From pivot/pivot_table

In [ ]:
multi_pivot = df.pivot_table(index=['City',"Month"], values='Sales')
multi_pivot

Sales
City    Month       
Karachi Feb    250.0
        Jan    200.0
Lahore  Feb    150.0
        Jan    100.0

###b) From concat with Keys

In [ ]:
jan = pd.DataFrame({"Product": ["A","B"], "Sales":[100,200]})
feb = pd.DataFrame({"Product": ["A","B"], "Sales":[150,250]})

multi_concat = pd.concat([jan, feb], keys=['Jan',"Feb"])
multi_concat

Product  Sales
Jan 0       A    100
    1       B    200
Feb 0       A    150
    1       B    250

###c) From groupby

In [ ]:
grouped = df.groupby(["City","Month"])['Sales'].sum()
grouped

City     Month
Karachi  Feb      250
         Jan      200
Lahore   Feb      150
         Jan      100
Name: Sales, dtype: int64

##3. Accessing data in MultiIndex
####a) Using .loc[]

In [ ]:
pivot_df

Sales
City    Month       
Karachi Feb    250.0
        Jan    200.0
Lahore  Feb    150.0
        Jan    100.0

In [ ]:
pivot_df.loc['Karachi', 'Jan']

,Karachi
,Jan
Sales,200.0


####Returns Sales value for that combination.

###b) Using .xs() (cross-section)

In [ ]:
pivot_df.xs("Lahore", level = 'City')

,Sales
Month,
Feb,150.0
Jan,100.0


####Returns sub-DataFrame/Series for the given level.

##4. Resetting MultiIndex
####Convert MultiIndex → columns:

In [ ]:
pivot_df

Sales
City    Month       
Karachi Feb    250.0
        Jan    200.0
Lahore  Feb    150.0
        Jan    100.0

In [ ]:
pivot_df_reset = pivot_df.reset_index()
pivot_df_reset

,City,Month,Sales
0,Karachi,Feb,250.0
1,Karachi,Jan,200.0
2,Lahore,Feb,150.0
3,Lahore,Jan,100.0


##5. Sorting MultiIndex
####Sort by index levels:

In [ ]:
pivot_df_sorted = pivot_df.sort_index(level=["City","Month"])
pivot_df_sorted

Sales
City    Month       
Karachi Feb    250.0
        Jan    200.0
Lahore  Feb    150.0
        Jan    100.0

#6. Wide vs Long Format
####1. What is Wide Format
In wide format, each variable forms a separate column.
Usually easy to read for humans but not always machine-learning friendly.

###Example: Monthly Sales (Wide)


###Characteristics:
| Product | Jan | Feb | Mar |
|---------|-----|-----|-----|
| A       | 100 | 120 | 130 |
| B       | 200 | 210 | 220 |

####1. Each month is a separate column → “wide” shape.
####2. Number of columns grows as variables or categories increase.
####3. Easy for pivoted reports, visual inspection.

##2. What is Long Format
####1. In long format, all measurements are stacked into a single column, with identifier columns specifying categories.
####2. Each row represents a single observation.
###Characteristics:
| Product | Month | Sales |
|---------|-------|-------|
| A       | Jan   | 100   |
| A       | Feb   | 120   |
| A       | Mar   | 130   |
| B       | Jan   | 200   |
| B       | Feb   | 210   |
| B       | Mar   | 220   |

####1. Number of rows grows instead of columns.
####2. Ideal for analysis, ML models, and plotting.
####3. Works well with groupby, pivot_table, seaborn, sklearn.

##3. When to Use Each Format
|Format|	Use Cases|
|------|-----------|
|Wide	|Dashboard, reporting, human-readable tables, quick Excel-style summary|
|Long	|Data analysis, plotting, ML preprocessing, statistical modeling|

Rule of thumb:

If you need grouping, aggregation, or feeding ML models, use long format.
If you need reporting or readability, wide format is fine.

##4. Why ML Prefers Long Format
####1. ML pipelines expect features as columns and observations as rows.
####2. Wide format can create many sparse columns → harder for algorithms like sklearn to handle.
####3. Long format is tidy, and pandas operations like groupby + agg work naturally.

#7. Reshaping with GroupBy + Unstack
##1. What is groupby().unstack()?

###It is a 2-step process:

###Step 1: groupby()
####Groups data and aggregates
###Step 2: .unstack()
####Converts grouped index into columns
###groupby + unstack = manual pivot table
####*Using groupby().unstack(), we can reshape aggregated data into a pivot-style table. This method provides more flexibility and control compared to pivot_table(), making it useful for advanced data transformations and analysis.*

##2. Create Dataset

In [1]:
import pandas as pd

df = pd.DataFrame({
    "City": ["Lahore","Lahore","Karachi","Karachi","Lahore","Karachi"],
    "Product": ["A","B","A","B","A","B"],
    "Sales": [100,200,150,300,120,180]
})

df

,City,Product,Sales
0,Lahore,A,100
1,Lahore,B,200
2,Karachi,A,150
3,Karachi,B,300
4,Lahore,A,120
5,Karachi,B,180


##3. Basic GroupBy

In [3]:
grouped = df.groupby(["City", "Product"])["Sales"].sum()
grouped

City     Product
Karachi  A          150
         B          480
Lahore   A          220
         B          200
Name: Sales, dtype: int64

##4. Apply .unstack()

In [4]:
table = grouped.unstack()
table

Product,A,B
City,,
Karachi,150,480
Lahore,220,200


##5. Fill Missing Values

In [5]:
table = grouped.unstack().fillna(0)
table

Product,A,B
City,,
Karachi,150,480
Lahore,220,200


##6. Alternative One-Line Version

In [15]:
df.groupby(['City', 'Product'])['Sales'].sum().unstack()

Product,A,B
City,,
Karachi,150,480
Lahore,220,200


##7. Comparison with pivot_table()

###Using Pivot_table

In [17]:
pd.pivot_table(df, values='Sales', index='City', columns='Product', aggfunc='sum')

Product,A,B
City,,
Karachi,150,480
Lahore,220,200


###Using groupby + unstack

In [18]:
df.groupby(['City','Product'])['Sales'].sum().unstack()

Product,A,B
City,,
Karachi,150,480
Lahore,220,200



####pivot_table = shortcut
####groupby + unstack = full control

##8. Multiple Aggregations

In [23]:
df.groupby(['City', 'Product']).agg({'Sales':['sum','mean']})

Sales       
                  sum   mean
City    Product             
Karachi A         150  150.0
        B         480  240.0
Lahore  A         220  110.0
        B         200  200.0

###With Unstack

In [22]:
df.groupby(['City', 'Product']).agg({'Sales':['sum','mean']}).unstack()

Sales                   
          sum        mean       
Product     A    B      A      B
City                            
Karachi   150  480  150.0  240.0
Lahore    220  200  110.0  200.0